# M2 lab: on or off? Unit commitment as a mixed-integer program

*Energy Systems, Step by Step* · Modeler tier · lab for lesson M2 · about 45 minutes, plus 90 minutes of practice at the end.

You will build the lesson's unit-commitment model in Python, solve its linear relaxation with `scipy.optimize.linprog(method="highs")` and the mixed-integer program with `scipy.optimize.milp` (both run the open-source HiGHS solver), grow the same branch-and-bound tree as the browser, fix the switches to read conditional prices, and settle the day. `assert` cells check that you get the browser's numbers.

**How to run it.** In Google Colab: *File → Upload notebook*, choose this file, then *Runtime → Run all* (Colab already has NumPy and SciPy). Locally: Python 3.9 or later with `numpy` and `scipy` (1.9 or later, for `milp`); `matplotlib` is optional, for the one plot. No other files are needed: the data are in the notebook.

**The data.** Prairie Town is fictional. Its plants, their switching data (minimum outputs, no-load and start-up costs, minimum run and down times) and the unserved-energy penalty are illustrative. Demand, wind and solar follow the real hourly shapes of 29 July 2025, the 2025 summer day with the highest hourly demand on the Southwest Power Pool grid (EIA Form EIA-930, Hourly Electric Grid Monitor), scaled to the town exactly as in lesson 0.

In [ ]:
import numpy as np
import scipy
from scipy.optimize import linprog, milp, LinearConstraint, Bounds

print("NumPy", np.__version__, "· SciPy", scipy.__version__)
major, minor = (int(x) for x in scipy.__version__.split(".")[:2])
assert (major, minor) >= (1, 9), "this lab needs SciPy 1.9 or later (pip install -U scipy)"

## 1. The day and the plants

Every number below is the same as in the browser lesson. Hours run 0 to 23; each lasts one hour, so a plant's output in MW over the hour is also its energy in MWh. The nuclear plant runs flat at 100 MW; wind and solar can be spilled; the two gas plants have on/off switches.

In [ ]:
HOURS = 24
demand = [337, 321, 311, 302, 298, 302, 310, 321, 338, 360, 384, 407, 423, 426, 442, 452, 449, 449, 445, 434, 420, 400, 380, 354]
wind   = [117, 116, 111, 107, 103, 102, 94, 81, 53, 40, 33, 30, 35, 42, 47, 50, 51, 55, 64, 75, 86, 105, 110, 109]
solar  = [0, 0, 0, 0, 0, 0, 1, 35, 125, 160, 181, 177, 173, 159, 160, 157, 163, 139, 111, 54, 4, 0, 0, 0]

NUCLEAR = dict(mw=100, cost=10)
# mw size, cost $/MWh, min minimum output when on (MW), noload $ per hour on, startup $ per start,
# up / down minimum run and down times (hours), init state before the day, end required state in the last hour
FLEET = {
    "cc": dict(mw=150, cost=40,  min=50, noload=300, startup=8000, up=6, down=6, init=1, end=1),   # gas combined-cycle
    "pk": dict(mw=100, cost=100, min=30, noload=300, startup=2000, up=2, down=2, init=0, end=None), # gas peaker
}
GAS = ["cc", "pk"]
PENALTY = 3000          # $ per MWh of unserved demand (illustrative, as in lesson 11)

need = [demand[t] - NUCLEAR["mw"] - wind[t] - solar[t] for t in range(HOURS)]
print("Need for gas, MW:", need)
assert max(demand) == 452 and demand.index(452) == 15
assert all(n % 5 == 0 for n in need)           # lesson 0's rounding: the need for gas is a multiple of 5 MW

## 2. The model

| Rule | In symbols | Rows per hour |
|---|---|---|
| Goal | $\min \sum_t \big( 10 \cdot 100 + \sum_g (c_g\,p_{g,t} + NL_g\,u_{g,t} + SU_g\,v_{g,t}) + V\,s_t \big)$ | 1 |
| Balance | $100 + r_t + \sum_g p_{g,t} + s_t = d_t$ | 1 |
| Output | $\underline{P}_g\,u_{g,t} \le p_{g,t} \le \bar P_g\,u_{g,t}$ | 2 per plant |
| Start and stop | $v_{g,t} - w_{g,t} = u_{g,t} - u_{g,t-1}$, with $u_{g,-1}$ the state before the day | 1 per plant |
| Minimum run | $\sum_{\tau=t-UT_g+1}^{t} v_{g,\tau} \le u_{g,t}$ | 1 per plant |
| Minimum down | $\sum_{\tau=t-DT_g+1}^{t} w_{g,\tau} \le 1 - u_{g,t}$ | 1 per plant |
| Switch | $u_{g,t} \in \{0, 1\}$ (the relaxation: $0 \le u_{g,t} \le 1$); $u_{\text{cc},23} = 1$ | bounds |

Symbols: $t$ hour; $g$ gas plant (cc, pk); $p$ output, MW; $r$ wind and solar used, at most what the weather gives, MW; $s$ unserved demand, MW; $u$ switch (1 on, 0 off); $v$ start-up and $w$ shut-down (1 in the hour it happens); $c$ running cost, \$/MWh; $NL$ no-load cost, \$ per hour on; $SU$ start-up cost, \$ per start; $\underline{P}$ minimum output and $\bar P$ size, MW; $UT$, $DT$ minimum run and down times, hours; $d$ demand, MW; $V$ the unserved-energy penalty, \$/MWh.

`build()` writes the model as a list of variables (bounds, cost, integer or not) and a list of rows (lower and upper limits), the same structure as the browser's `build()`. The helpers then hand it to `linprog` (for an LP) or `milp` (for the mixed-integer program).

In [ ]:
ALL = dict(startup=True, pmin=True, minrun=True)

def build(dem=None, onoff=False, relax=False, rules=ALL, fix=None, add=None, fleet=None, day=None):
    # onoff=False: M1's dispatch LP (no switches). fix: {"u_pk_17": 1, ...} fixes switches. add: (hour, MWh) extra demand.
    dem = list(dem or demand); fleet = fleet or FLEET
    wnd, sol = (day or (wind, solar))
    if add: dem[add[0]] += add[1]
    V, R = [], []                                    # variables: (name, lb, ub, cost, integer); rows: (name, {var: coef}, lo, hi)
    def var(n, lo, hi, c=0.0, i=0): V.append((n, lo, hi, c, i))
    for t in range(HOURS):
        bal = {f"nuc_{t}": 1, f"ren_{t}": 1, f"shed_{t}": 1}
        var(f"nuc_{t}", NUCLEAR["mw"], NUCLEAR["mw"], NUCLEAR["cost"])
        var(f"ren_{t}", 0, wnd[t] + sol[t])
        var(f"shed_{t}", 0, np.inf, PENALTY)
        for g in GAS:
            u, p = fleet[g], f"p_{g}_{t}"
            var(p, 0, u["mw"], u["cost"]); bal[p] = 1
            if not onoff:
                continue
            on, st, sd = f"u_{g}_{t}", f"v_{g}_{t}", f"w_{g}_{t}"
            lo, hi = 0, 1
            if t == HOURS - 1 and u["end"] is not None: lo = hi = u["end"]
            if fix and on in fix: lo = hi = fix[on]
            var(on, lo, hi, u["noload"] if rules["startup"] else 0, 0 if relax else 1)
            var(st, 0, 1, u["startup"] if rules["startup"] else 0)
            var(sd, 0, 1)
            R.append((f"max_{g}_{t}", {p: 1, on: -u["mw"]}, -np.inf, 0))
            if rules["pmin"]:
                R.append((f"min_{g}_{t}", {p: 1, on: -u["min"]}, 0, np.inf))
            logic = {on: 1, st: -1, sd: 1}
            if t > 0: logic[f"u_{g}_{t-1}"] = -1
            rhs = 0 if t > 0 else u["init"]
            R.append((f"logic_{g}_{t}", logic, rhs, rhs))
            if rules["minrun"] and u["up"] > 1:
                row = {on: 1}
                for s in range(max(0, t - u["up"] + 1), t + 1): row[f"v_{g}_{s}"] = row.get(f"v_{g}_{s}", 0) - 1
                R.append((f"up_{g}_{t}", row, 0, np.inf))
            if rules["minrun"] and u["down"] > 1:
                row = {on: 1}
                for s in range(max(0, t - u["down"] + 1), t + 1): row[f"w_{g}_{s}"] = row.get(f"w_{g}_{s}", 0) + 1
                R.append((f"down_{g}_{t}", row, -np.inf, 1))
        R.append((f"bal_{t}", bal, dem[t], dem[t]))
    return V, R

def matrix(V, R):
    col = {v[0]: j for j, v in enumerate(V)}
    A = np.zeros((len(R), len(V)))
    for i, (_, terms, _, _) in enumerate(R):
        for n, a in terms.items(): A[i, col[n]] += a
    return A, col

def solve_lp(V, R):
    # linprog wants A_ub x <= b_ub and A_eq x = b_eq: equality rows go to A_eq (their duals are res.eqlin.marginals)
    A, col = matrix(V, R)
    eq = [i for i, r in enumerate(R) if r[2] == r[3]]
    ub = [(i, 1) for i, r in enumerate(R) if r[2] != r[3] and np.isfinite(r[3])] + [(i, -1) for i, r in enumerate(R) if r[2] != r[3] and np.isfinite(r[2])]
    res = linprog([v[3] for v in V], A_ub=np.array([s * A[i] for i, s in ub]) if ub else None,
                  b_ub=np.array([s * (R[i][3] if s > 0 else R[i][2]) for i, s in ub]) if ub else None,
                  A_eq=A[eq], b_eq=[R[i][2] for i in eq], bounds=[(v[1], None if v[2] == np.inf else v[2]) for v in V], method="highs")
    out = dict(status=res.status, ok=res.status == 0, cost=res.fun if res.status == 0 else None)
    if res.status == 0:
        out["x"] = {v[0]: res.x[j] for j, v in enumerate(V)}
        out["dual"] = {R[i][0]: m + 0.0 for i, m in zip(eq, res.eqlin.marginals)}   # d(cost)/d(right-hand side)
    return out

def solve_milp(V, R):
    A, col = matrix(V, R)
    res = milp([v[3] for v in V], constraints=LinearConstraint(A, [r[2] for r in R], [r[3] for r in R]),
               integrality=[v[4] for v in V], bounds=Bounds([v[1] for v in V], [v[2] for v in V]))
    out = dict(status=res.status, ok=res.status == 0, cost=res.fun, res=res)
    if res.status == 0: out["x"] = {v[0]: res.x[j] for j, v in enumerate(V)}
    return out

def col(sol, prefix): return [sol["x"][f"{prefix}_{t}"] for t in range(HOURS)]

def clean(a):
    # plain Python numbers for printing and comparing: whole numbers as int, others rounded to 6 decimals
    out = []
    for x in a:
        x = round(float(x), 6) + 0.0
        out.append(int(x) if x.is_integer() else x)
    return out

V, R = build(onoff=True)
print(len(V), "decisions,", sum(v[4] for v in V), "of them switches;", len(R), "rows")
assert (len(V), len(R), sum(v[4] for v in V)) == (264, 264, 48)

## 3. Step 1: M1's dispatch model on the summer peak

No switches: each gas plant may run anywhere from 0 to its size. The 24 balance duals are the hourly prices.

In [ ]:
lp = solve_lp(*build())
pk_lp = clean(col(lp, "p_pk"))
prices_lp = clean(lp["dual"][f"bal_{t}"] for t in range(HOURS))
print(f"Cost of the day: ${lp['cost']:,.0f}")
print("Peaker, MW:", pk_lp)
print("Prices, $/MWh:", prices_lp)

assert abs(lp["cost"] - 159500) < 1e-6
assert pk_lp[17:23] == [5, 20, 55, 80, 45, 20]                   # 5 MW at 17:00: below the peaker's 30 MW minimum
assert prices_lp == [100 if 17 <= t <= 22 else 40 for t in range(HOURS)]
print("Step 1 matches the browser.")

## 4. Step 2: add the rules, but let the switches slide (the LP relaxation)

Add start-up and no-load costs, then minimum outputs, then minimum run and down times, with every switch allowed anywhere from 0 to 1.

In [ ]:
steps = [dict(startup=True, pmin=False, minrun=False), dict(startup=True, pmin=True, minrun=False), ALL]
for rules in steps:
    rx = solve_lp(*build(onoff=True, relax=True, rules=rules))
    sw = clean(col(rx, "u_pk"))
    print(f"{rules}: ${rx['cost']:,.0f}; peaker switches 17:00-22:00 {sw[17:23]}")
    assert abs(rx["cost"] - 168975) < 1e-6 and sw[17:23] == [0.05, 0.2, 0.55, 0.8, 0.45, 0.2]

starts = sum(col(rx, "v_pk"))
print(f"The relaxation pays {starts:.2f} of a peaker start; at 17:00 the peaker is {sw[17]:.0%} on and makes {rx['x']['p_pk_17']:.0f} MW")
assert abs(starts - 0.8) < 1e-6 and all(abs(x - 1) < 1e-9 for x in col(rx, "u_cc"))
relaxed = rx

**Why not round?** Round each switch to the nearest whole number, keep the combined-cycle plant on, and re-solve with the switches fixed.

In [ ]:
def plan_fix(peaker_on):
    fix = {f"u_cc_{t}": 1 for t in range(HOURS)}
    fix.update({f"u_pk_{t}": int(peaker_on[t]) for t in range(HOURS)})
    return fix

rounded = [int(round(x)) for x in col(relaxed, "u_pk")]
r = solve_lp(*build(onoff=True, relax=True, fix=plan_fix(rounded)))
print("Rounded: peaker on in hours", [t for t in range(HOURS) if rounded[t]], f"-> {sum(col(r, 'shed')):.0f} MWh unserved, ${r['cost']:,.0f}")
assert [t for t in range(HOURS) if rounded[t]] == [19, 20]
assert abs(sum(col(r, "shed")) - 90) < 1e-6 and abs(r["cost"] - 430300) < 1e-6

## 5. Step 4: branch and bound

The same rules as the browser: open the node with the lowest bound (ties: the older node); branch on the switch closest to half on (ties: the earlier hour, then the combined-cycle plant); solve both children, "off" first. A node is pruned when its bound is no better than the best plan found; a node whose switches are all whole is a plan.

In [ ]:
TOL = 1e-6

def fractional(sol):
    return [(f"u_{g}_{t}", sol["x"][f"u_{g}_{t}"]) for t in range(HOURS) for g in GAS
            if abs(sol["x"][f"u_{g}_{t}"] - round(sol["x"][f"u_{g}_{t}"])) > TOL]

def branch_and_bound(verbose=True):
    nodes = [dict(id=0, parent=None, fix={}, status="pending")]
    best, bounds_after = None, []
    def accept(n):
        nonlocal best
        s = solve_lp(*build(onoff=True, relax=True, fix=n["fix"]))
        n["bound"], n["frac"], n["shed"] = s["cost"], fractional(s), sum(col(s, "shed"))
        if best is not None and n["bound"] >= best["bound"] - TOL: n["status"] = "pruned"
        elif not n["frac"]:
            n["status"] = "plan"
            if best is None or n["bound"] < best["bound"] - TOL:
                best = n
                for m in nodes:
                    if m["status"] == "open" and m["bound"] >= n["bound"] - TOL: m["status"] = "pruned"
        else: n["status"] = "open"
    accept(nodes[0])
    def best_bound():
        return min([m["bound"] for m in nodes if m["status"] == "open"] + [best["bound"] if best else np.inf])
    bounds_after.append(best_bound())
    while any(m["status"] == "open" for m in nodes):
        n = min((m for m in nodes if m["status"] == "open"), key=lambda m: (m["bound"], m["id"]))
        name, x = n["frac"][0]
        for f in n["frac"][1:]:
            if abs(f[1] - 0.5) < abs(x - 0.5) - TOL: name, x = f
        n["status"], n["on"] = "branched", (name, round(x, 6))
        for val in (0, 1):
            c = dict(id=len(nodes), parent=n["id"], fix={**n["fix"], name: val}, status="pending", by=(name, val))
            nodes.append(c); accept(c)
        bounds_after.append(best_bound())
        if verbose:
            kids = nodes[-2:]
            print(f"branch on {name} = {x:.2f}:  off ${kids[0]['bound']:,.0f} ({kids[0]['status']}, {kids[0]['shed']:.0f} MWh unserved)"
                  f"  on ${kids[1]['bound']:,.0f} ({kids[1]['status']})")
    return nodes, best, bounds_after

nodes, best, bounds_after = branch_and_bound()
print(f"{len(nodes)} LPs; best plan ${best['bound']:,.0f}; best bound after each branching {[round(b) for b in bounds_after]}")

# the browser's tree
assert [round(n["bound"]) for n in nodes] == [168975, 328920, 169510, 300695, 169735, 242160, 170575, 228515, 171415, 185900, 173200]
assert [n["on"] for n in nodes if "on" in n] == [("u_pk_19", 0.55), ("u_pk_21", 0.45), ("u_pk_18", 0.2), ("u_pk_22", 0.2), ("u_pk_17", 0.05)]
assert [round(b) for b in bounds_after] == [168975, 169510, 169735, 170575, 171415, 173200]
assert round(best["bound"]) == 173200 and nodes[9]["status"] == "plan" and round(nodes[9]["shed"]) == 5
print("The tree matches the browser.")

## 6. Step 5: the mixed-integer program in one call

`milp` runs HiGHS's own branch and cut: presolve, cutting planes and heuristics on top of the same search. Its result carries no duals.

In [ ]:
mi = solve_milp(*build(onoff=True))
on_pk = [int(round(x)) for x in col(mi, "u_pk")]
print(f"MILP: ${mi['cost']:,.0f}, gap {mi['res'].mip_gap:.2%}; peaker on in hours {[t for t in range(HOURS) if on_pk[t]]}")
print("Peaker, MW, 17:00-22:00:", clean(col(mi, "p_pk"))[17:23])
print("Duals in the result?", hasattr(mi["res"], "eqlin"))

assert abs(mi["cost"] - 173200) < 1e-6 and mi["res"].mip_gap <= 1e-9
assert [t for t in range(HOURS) if on_pk[t]] == list(range(17, 23))
assert clean(col(mi, "p_pk"))[17:23] == [30, 30, 55, 80, 45, 30]
assert not hasattr(mi["res"], "eqlin")

## 7. Step 6: what does one more MWh cost?

Add 0 to 10 MWh at 23:00 and solve the mixed-integer program each time. Compare the extra cost of the day with \$40 per MWh.

In [ ]:
stair = clean(solve_milp(*build(onoff=True, add=(23, x)))["cost"] - mi["cost"] for x in range(11))
print("Extra cost of the day, $:", stair)
print("Cost of each added MWh, $:", [round(stair[x] - stair[x - 1]) for x in range(1, 11)])
assert stair == [0, 40, 80, 120, 160, 200, 2340, 2380, 2420, 2460, 2500]
assert round(stair[6] - stair[5]) == 2140          # the 6th MWh keeps the peaker on at 23:00

## 8. Step 7: fix the switches and re-price

Fix every switch at its MILP value and solve the LP that is left. Its balance duals are prices *conditional on this commitment*. Check two of them by adding 1 MWh with the switches still fixed.

In [ ]:
fix = {f"u_{g}_{t}": int(round(mi["x"][f"u_{g}_{t}"])) for g in GAS for t in range(HOURS)}
fx = solve_lp(*build(onoff=True, relax=True, fix=fix))
cond = clean(fx["dual"][f"bal_{t}"] for t in range(HOURS))
print(f"Fixed LP: ${fx['cost']:,.0f}; conditional prices {cond}")
assert abs(fx["cost"] - 173200) < 1e-6
assert cond == [100 if 19 <= t <= 21 else 40 for t in range(HOURS)]

for t in (18, 20):
    up = solve_lp(*build(onoff=True, relax=True, fix=fix, add=(t, 1)))
    print(f"+1 MWh at {t}:00 with the switches fixed: +${up['cost'] - fx['cost']:.2f}")
    assert abs(up["cost"] - fx["cost"] - cond[t]) < 1e-6

## 9. Step 8: settle the day

Pay each gas plant the conditional price for every MWh it made, and compare with what it spent: running cost, no-load cost for every hour on, start-up cost for every start.

In [ ]:
def settle(sol, prices, fleet=None):
    fleet = fleet or FLEET
    out = {}
    for g in GAS:
        p, u, v = col(sol, f"p_{g}"), col(sol, f"u_{g}"), col(sol, f"v_{g}")
        revenue = sum(prices[t] * p[t] for t in range(HOURS))
        running = fleet[g]["cost"] * sum(p)
        noload = fleet[g]["noload"] * round(sum(u))
        startup = fleet[g]["startup"] * round(sum(v))
        out[g] = dict(revenue=revenue, running=running, noload=noload, startup=startup, profit=revenue - running - noload - startup)
    return out

s = settle(mi, cond)
for g in GAS:
    print(g, {k: round(v) for k, v in s[g].items()})
uplift = max(0, -s["pk"]["profit"])
print(f"The peaker is short ${uplift:,.0f}: the uplift (make-whole) payment")
assert [round(s["pk"][k]) for k in ("revenue", "running", "noload", "startup")] == [21600, 27000, 1800, 2000]
assert round(uplift) == 9200 and round(s["cc"]["profit"]) == 19800

In [ ]:
# Optional: the peaker's switch in the relaxation and in the mixed-integer program (needs matplotlib)
try:
    import matplotlib.pyplot as plt
except ImportError:
    plt = None
    print("matplotlib is not installed: skipping the plot")
if plt is not None:
    fig, ax = plt.subplots(figsize=(8, 3))
    hrs = np.arange(HOURS)
    ax.bar(hrs - 0.2, col(relaxed, "u_pk"), width=0.4, label="relaxation (LP)", color="tab:orange")
    ax.bar(hrs + 0.2, on_pk, width=0.4, label="commitment (MILP)", color="tab:red")
    ax.set_xlabel("Hour"); ax.set_ylabel("Peaker switch"); ax.set_title("29 July 2025: the peaker's switch")
    ax.legend(); plt.show()

## 10. Practice (about 90 minutes)

Write your own cells below each question. Predict first, then solve.

1. **The sunny day.** On 30 May 2025 (the arrays in the next cell, scaled the same way) the need for gas falls below zero at midday. Drop the combined-cycle plant's end-of-day rule (`end=None`), solve the mixed-integer program, and read its switches. Then sweep its start-up cost from \$2,000 to \$20,000: at what cost does it stop switching off over the solar trough? Explain the threshold with its no-load cost and the energy it would otherwise make at its minimum.
2. **Branching rules.** Change the branching rule to "the earliest fractional hour" and then to "the switch closest to 1". How many LPs does each tree need? Why can a different rule give a different tree, but never a different optimal cost?
3. **Ramps.** Add M1's ramp rule for the combined-cycle plant ($-40 \le p_{\text{cc},t} - p_{\text{cc},t-1} \le 40$, starting from 115 MW). Which conditional prices change, and why? Check one by a finite difference.
4. **Reserves.** Require 20 MW of spare headroom every hour: $\sum_g (\bar P_g u_{g,t} - p_{g,t}) \ge 20$. How do the switches, the cost and the peaker's uplift change?
5. **Several optimal relaxations.** Set the peaker's minimum run time to 3 hours. Solve the root relaxation twice, once with the costs of every variable changed by a tiny random amount. Do the switches change while the cost does not? What does this mean for reproducing a branch-and-bound tree with another solver?
6. **Other prices.** Use the root relaxation's balance duals as prices instead and settle the day again. Does the peaker still need an uplift payment? Who pays more for energy, and who less? (This is where pricing designs differ; it is a Researcher-tier question.)

In [ ]:
# Data for practice question 1: 30 May 2025 (SPP shapes from EIA-930, scaled as lesson 0)
demand_may = [223, 218, 214, 211, 213, 218, 225, 233, 242, 248, 255, 261, 268, 274, 283, 287, 294, 295, 290, 282, 271, 264, 255, 239]
wind_may   = [23, 23, 24, 26, 33, 38, 41, 39, 20, 13, 13, 11, 11, 12, 12, 11, 12, 13, 12, 13, 19, 29, 40, 49]
solar_may  = [0, 0, 0, 0, 0, 0, 4, 59, 137, 165, 177, 175, 172, 172, 166, 171, 162, 147, 123, 74, 12, 0, 0, 0]
print("Need for gas, MW:", [demand_may[t] - 100 - wind_may[t] - solar_may[t] for t in range(HOURS)])
# example call: solve_milp(*build(dem=demand_may, day=(wind_may, solar_may), onoff=True, fleet={**FLEET, "cc": {**FLEET["cc"], "end": None}}))